# Demo: Building a Custom Retriever
### Module 5, Topic 3 — RAG from Scratch

**What you'll see in this notebook:**
1. Embed the Naija One Bank chunks from Topic 2
2. Store each chunk alongside its embedding
3. Embed a new question and compare it against every stored chunk
4. Rank chunks by similarity and return the top-k
5. Package all of this into a single reusable `retrieve()` function

**A quick note on the embedding model:** Claude doesn't generate embeddings itself — it's a text generation model, not a vector model. Anthropic's recommended embedding partner is **Voyage AI**, so that's what we'll use here alongside the Anthropic SDK from Topic 1.


## Step 0 — Install Voyage AI

Voyage AI offers a free tier that's enough for this demo.

In [ ]:
!pip install voyageai --quiet

## Step 1 — Set Up the Embedding Client

Same pattern as Topic 1's Anthropic client — API key from an environment variable, never hardcoded.

In [ ]:
import os
import voyageai

vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))
EMBED_MODEL = "voyage-4"

print("Embedding client ready.")

## Step 2 — The Chunks From Topic 2

These are the exact sentence-aware chunks produced in the previous demo. We're picking up right where Topic 2 left off.

In [ ]:
chunks = [
    "Naija One Bank — Flexi Save Account Policy (Effective 2026)\n\nThe Flexi Save account is Naija One Bank's flagship savings product for individual customers.",
    "It is designed for customers who want easy access to their funds while still earning competitive interest.",
    "The account has no monthly maintenance fee as long as the minimum balance is maintained.",
    "Interest is calculated daily and credited monthly at a rate of 4.2% per annum.",
    "The minimum opening balance required to activate the account is NGN 5,000.",
    "To continue earning interest, customers must maintain a minimum balance of NGN 1,000 at all times.",
    "Customers are permitted 3 free withdrawals per month. A fee of NGN 500 applies to each withdrawal beyond this limit.",
    "Withdrawals can be made via the mobile app, at any branch, or through an ATM using the Flexi Save debit card.",
    "Accounts that fall below the minimum balance for more than 60 consecutive days will be automatically converted to a Basic Save account, which does not earn interest.",
    "Customers can reactivate Flexi Save status by restoring the minimum balance.",
]

print(f"Number of chunks: {len(chunks)}")

## Step 3 — Embed and Store the Chunks

Each chunk is embedded once, using `input_type="document"` — Voyage tunes the embedding differently depending on whether text is a document to be searched or a query doing the searching.

In [ ]:
result = vo.embed(chunks, model=EMBED_MODEL, input_type="document")
chunk_embeddings = result.embeddings

# Store each chunk alongside its embedding
knowledge_base = list(zip(chunks, chunk_embeddings))

print(f"Stored {len(knowledge_base)} (chunk, embedding) pairs.")
print(f"Each embedding has {len(chunk_embeddings[0])} numbers.")

## Step 4 — Cosine Similarity, From Scratch

This is the one piece of math in this notebook: a function that scores how close two vectors are.

In [ ]:
import numpy as np

def cosine_similarity(vec_a, vec_b):
    vec_a = np.array(vec_a)
    vec_b = np.array(vec_b)
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

# Sanity check: a vector compared to itself should score 1.0
print(cosine_similarity(chunk_embeddings[0], chunk_embeddings[0]))

## Step 5 — Embed a Question

Notice the words in this question barely overlap with the words in the chunks — that's the point.

In [ ]:
question = "How much can I take out of my account each month before I get charged?"

query_result = vo.embed([question], model=EMBED_MODEL, input_type="query")
query_embedding = query_result.embeddings[0]

print("Query embedded.")

## Step 6 — Score Every Chunk Against the Question

In [ ]:
scores = []
for chunk_text, chunk_vec in knowledge_base:
    score = cosine_similarity(query_embedding, chunk_vec)
    scores.append((chunk_text, score))

for chunk_text, score in scores:
    print(f"{score:.4f}  {chunk_text[:70]}")

## Step 7 — Rank and Return the Top-K

Sort by score, highest first, and keep only the top 3.

In [ ]:
ranked = sorted(scores, key=lambda pair: pair[1], reverse=True)
top_k = ranked[:3]

print("Top 3 chunks for this question:\n")
for i, (chunk_text, score) in enumerate(top_k):
    print(f"#{i+1} (score: {score:.4f})")
    print(chunk_text)
    print()

## Step 8 — Check the Result

The withdrawal-limit chunk ("3 free withdrawals per month... NGN 500 fee") should rank at or near the top — even though the question said "take out" and "charged," not "withdrawal" and "fee." That's the embedding model matching on meaning, not exact words, exactly as described in the slides.

## Step 9 — Package It Into a Reusable Retriever

Everything above, wrapped into one function that takes a question and returns the top-k most relevant chunks.

In [ ]:
def retrieve(query, knowledge_base, k=3):
    query_embedding = vo.embed([query], model=EMBED_MODEL, input_type="query").embeddings[0]

    scores = []
    for chunk_text, chunk_vec in knowledge_base:
        score = cosine_similarity(query_embedding, chunk_vec)
        scores.append((chunk_text, score))

    ranked = sorted(scores, key=lambda pair: pair[1], reverse=True)
    return ranked[:k]

print("retrieve() is ready.")

## Step 10 — Try It on a New Question

Let's test the retriever with a completely different question, one it hasn't seen tuned for.

In [ ]:
new_question = "What happens if my balance drops too low for two months?"

results = retrieve(new_question, knowledge_base, k=3)

print(f"Question: {new_question}\n")
for i, (chunk_text, score) in enumerate(results):
    print(f"#{i+1} (score: {score:.4f})")
    print(chunk_text)
    print()

## Step 11 — Check the Result

The chunk about accounts falling below the minimum balance for "more than 60 consecutive days" and being converted to a Basic Save account should rank first — the retriever correctly matched "too low for two months" (roughly 60 days) to that chunk, again without any shared exact wording.

## What's Next

We now have a working, reusable `retrieve()` function. It's built entirely from scratch — an embedding call, a cosine similarity function, and a sort. In Topic 4, we'll see how **LangChain** wraps this exact same logic into a few lines using its own retriever abstractions, and compare it directly to what we just built here. Then in Topic 5, this `retrieve()` function gets wired directly into an LLM call — completing the RAG pipeline.